# 1.5 - Exploration et nettoyage de données réelles

**Navigation** : [<< 1.4 Visualisation](1.4-Visualisation_Matplotlib_Seaborn.ipynb) | [Index](../README.md) | [>> 1.6 Statistiques](1.6-Statistiques_Descriptives.ipynb)

**Durée indicative** : ~75 min.

**Jeu de données** : `data/penguins_raw.csv` — l'export **brut** de terrain des manchots de la station Palmer (vraies cases vides, vraies catégories bavardes ; dimensions mesurées dans l'audit ci-dessous). Provenance et licence : `data/PROVENANCE.md`.


## Objectifs d'apprentissage

1. Mener l'**audit de première rencontre** : dimensions, types, valeurs manquantes, doublons, plages de valeurs.
2. Nettoyer **en consignant** : chaque décision de nettoyage entre au **journal des décisions** (problème, décision, justification, lignes concernées).
3. Distinguer les problèmes **structurels** (colonne quasi vide), **de saisie** (catégories verbeuses, casse) et **d'observation** (mesures manquantes, valeurs extrêmes).
4. Reconnaître un **faux doublon** (collision de clé) et un **quasi-doublon** (mesures identiques) — et statuer.
5. Comprendre que la table propre **conserve** les valeurs manquantes quand l'analyse les **filtre** : nettoyer n'est pas supprimer.

## Pourquoi un journal

Un nettoyage non documenté est un nettoyage non reproductible : le lecteur — votre binôme, votre correcteur, l'agent que vous orchestrerez dans les labs — ne peut plus dire ce que la table analysée contient de différent de la table source. Le journal des décisions est le contrat : chaque transformation y figure avec sa justification. Nous le tiendrons dans un tableau qui grossit à chaque décision, et il sera le **livrable final** de ce notebook.


In [1]:
import pandas as pd

brut = pd.read_csv("../data/penguins_raw.csv")
print("dimensions :", brut.shape)
brut.head(3)


dimensions : (344, 17)


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN


## 1. Première rencontre : audit

Avant toute transformation, trois questions : **combien** de lignes et colonnes, de **quels types**, avec **quelles valeurs** plausibles ? `info()` donne les deux premières réponses en un geste — nombre de non-nuls par colonne inclus.

On ne touche **rien** avant d'avoir posé ce diagnostic : nettoyer un jeu qu'on n'a pas encore regardé, c'est décider à l'aveugle. L'audit est le seul moment où l'on regarde la table **telle qu'elle est sortie de terrain** — c'est donc là que se forment les hypothèses (quelles colonnes sont des catégories, lesquelles sont verbeuses, lesquelles sont quasi vides) que le reste du notebook va confirmer ou infirmer, ligne par ligne.

In [2]:
brut.info()


<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   studyName            344 non-null    str    
 1   Sample Number        344 non-null    int64  
 2   Species              344 non-null    str    
 3   Region               344 non-null    str    
 4   Island               344 non-null    str    
 5   Stage                344 non-null    str    
 6   Individual ID        344 non-null    str    
 7   Clutch Completion    344 non-null    str    
 8   Date Egg             344 non-null    str    
 9   Culmen Length (mm)   342 non-null    float64
 10  Culmen Depth (mm)    342 non-null    float64
 11  Flipper Length (mm)  342 non-null    float64
 12  Body Mass (g)        342 non-null    float64
 13  Sex                  333 non-null    str    
 14  Delta 15 N (o/oo)    330 non-null    float64
 15  Delta 13 C (o/oo)    331 non-null    float64
 16  C

**Lecture de l'audit.** Déjà beaucoup : `Species` et `Sex` sont du texte (donc des **catégories**), `Date Egg` est une date stockée en `object` (texte — à convertir), `Comments` n'a que 54 valeurs non nulles sur 344 (290 manquantes : quasi vide), et les colonnes de mesures ont toutes 342 non-nuls — **deux individus** échappent à toutes les mesures. Les isotopes `Delta 15 N` / `Delta 13 C` manquent pour ~13-14 individus. Ce premier balayage oriente **tout** le nettoyage qui suit : chaque constat pointé ici devient, plus bas, une section — les catégories verbeuses mènent à la section 5, les 290 de `Comments` et les 11 de `Sex` à la section 3, les deux individus sans mesures aussi, les plages de valeurs à la section 6. C'est la force d'un audit complet : il ne se contente pas de **décrire** les données, il **programme** le travail. On sait, avant de toucher à quoi que ce soit, exactement quelles sections le carnet va devoir contenir — et on n'est jamais pris au dépourvu, parce que tout a été vu d'abord.

**L'audit est une check-list, pas un réflexe.** Pourquoi cette liste précise ? Cinq sondes, et elles forment une **clôture** : **combien** (dimensions), **quelles natures** (types), **quoi de manquant** (manquants), **quoi de répété** (doublons), **quoi d'extrême** (plages de valeurs). Chaque décision de nettoyage qui suit dans le carnet se rattache à l'une d'elles — aucune transformation ne surgit de nulle part. C'est ce qui fait de l'audit un **contrat** plutôt qu'un réflexe : on ne « regarde un peu » les données, on les passe en revue **complète**, et on note ce que chaque sonde révèle. Un nettoyage dont on ne peut pas dire « cette décision répondait à cette sonde de l'audit » est un nettoyage à l'instinct — et l'instinct, sur des données réelles, est précisément là qu'on se trompe.

## 2. Le journal des décisions

On le remplit au fil du notebook avec une fonction d'une ligne — le geste compte plus que l'outil. Chaque entrée porte **quatre champs** : le problème détecté, la décision prise, la justification, et les lignes concernées (un compte, pas une liste — elle peut être longue).

La discipline tient dans l'**ordre** : on **observe** d'abord (la cellule au-dessus produit la mesure), puis on **décide**, puis on **consigne** — jamais l'inverse. Une entrée sans la mesure qui la motive est un aveu ; une décision sans justification est une opinion. Le journal n'est pas un compte-rendu rédigé après coup, c'est le fil rouge qui rend le nettoyage **auditable** : à la fin, on lira le journal, pas le code.

In [3]:
journal = []

def consigner(etape, probleme, decision, justification, lignes):
    """Ajoute une decision au journal des decisions de nettoyage."""
    journal.append({
        "etape": etape, "probleme": probleme, "decision": decision,
        "justification": justification, "lignes_concernees": lignes,
    })
    print(f"[journal] {etape} : {decision}")

print("journal initialise :", len(journal), "entree(s)")


journal initialise : 0 entree(s)


## 3. Valeurs manquantes : trois problèmes différents

Le compte brut d'abord, trié — puis la décision, colonne par colonne. La règle de lecture : une valeur manquante n'a pas toujours le même statut. `Comments` vide est **structurel** (la note n'existe que pour documenter une anomalie). `Sex` inconnu est une **incertitude d'observation** (le terrain n'a pas pu sexer l'oiseau). Une mesure manquante est une **donnée perdue** pour cette analyse.

Ce que cette section va montrer, c'est que le **compte** de manquants ne suffit jamais à décider : deux colonnes peuvent afficher le même chiffre « 2 » et cacher deux situations opposées (deux cases vides par colonne, ou deux individus sans **aucune** mesure). Le chiffre **oriente**, les **lignes** **décident**. On applique donc partout la même méthode : compter, puis ouvrir les lignes, puis statuer — et consigner.

In [4]:
manquantes = brut.isna().sum().sort_values(ascending=False)
manquantes = manquantes[manquantes > 0]
manquantes


Comments               290
Delta 15 N (o/oo)       14
Delta 13 C (o/oo)       13
Sex                     11
Culmen Length (mm)       2
Flipper Length (mm)      2
Culmen Depth (mm)        2
Body Mass (g)            2
dtype: int64

**Lecture du décompte.** La liste triée se range en trois étages, et chaque étage est un **type** de manque différent, pas trois chiffres de même nature. Premier étage : `Comments`, 290 manquants sur 344 — 84 % de la colonne vide. Un taux pareil n'est pas des données perdues, c'est une **colonne qui n'existe que pour l'exception** : la note n'est écrite que quand quelque chose cloche, donc son vide est la règle, pas l'anomalie. Second étage : les deux isotopes `Delta 15 N` et `Delta 13 C`, 14 et 13 manquants — un sous-ensemble d'oiseaux n'a tout simplement pas été échantillonné pour l'analyse des isotopes ; le vide est ici **programmé** par le protocole de terrain. Troisième étage : `Sex` (11) et les quatre mesures morphométriques (2 chacune). C'est ici le piège de lecture : le « 2 » répété sur **quatre** colonnes n'est pas 2 × 4 = 8 valeurs manquantes dispersées, ce sont **les mêmes deux oiseaux** qui manquent à **toutes** les mesures à la fois — la cellule suivante va le prouver en croisant les colonnes. Leçon de méthode : un compte de manquants **oriente**, il ne **décide** jamais ; pour savoir si c'est « beaucoup de cases vides » ou « quelques individus incomplets », il faut ouvrir les lignes.

In [5]:
# Colonne quasi vide : structurelle, on la retire de la table de travail
consigner(
    etape="comments",
    probleme="Comments quasi vide (la note n'existe que pour signaler une anomalie de saisie)",
    decision="retirer la colonne de la table de travail",
    justification="colonne de commentaires libres, pas une variable analysable ; les anomalies documentees sont citees dans les etapes qui suivent",
    lignes=344,
)
brut = brut.drop(columns=["Comments"])
print("dimensions apports :", brut.shape)


[journal] comments : retirer la colonne de la table de travail
dimensions apports : (344, 16)


In [6]:
# Deux individus sans AUCUNE mesure morphometrique
sans_mesures = brut[brut["Culmen Length (mm)"].isna() &
                   brut["Culmen Depth (mm)"].isna() &
                   brut["Flipper Length (mm)"].isna() &
                   brut["Body Mass (g)"].isna()]
sans_mesures


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo)
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,PAL0910,120,Gentoo penguin (Pygoscelis papua),Anvers,Biscoe,"Adult, 1 Egg Stage",N38A2,No,2009-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Lecture — garder, c'est statuer.** Regardez les deux lignes que la cellule vient d'isoler : un Adelie de Torgersen, un Gentoo de Biscoe, **toutes les quatre mesures au NaN**. Ce ne sont pas des lignes corrompues, ce sont deux **oiseaux réels** que le terrain n'a pas mesurés — et leur espèce, leur île, leur stade, leur date de ponte sont **vraies**. La tentation serait de les `dropna()` ; on s'en abstient, et c'est une **décision** qu'on consigne, pas un défaut. Concrètement, on ne les supprime **nulle part** : la table les garde, et ce sont les seules analyses morphométriques qui les mettent à l'écart, au moment du calcul, avec un `dropna(subset=[...])` ciblé. Résultat : le comptage des effectifs par espèce reste **stable** quelle que soit l'analyse — 152 Adelie, 124 Gentoo, 68 Chinstrap — au lieu de dépendre du sous-ensemble qu'on regarde. C'est le premier exemple, en situation, du principe « la table **garde**, l'analyse **filtre** ».

In [7]:
# Decision : la table propre CONSERVE ces lignes (ce sont des oiseaux reels, jamais mesures — sexes inconnus),
# les analyses morphometriques les filtreront au moment de calculer.
consigner(
    etape="mesures-absentes",
    probleme="2 individus sans aucune mesure (bec, aile, masse)",
    decision="conserver les lignes dans la table propre ; les analyses morphometriques utilisent dropna(subset=[...]) a l'usage",
    justification="l'information Espece/Ile reste valable (Sex est NA pour ces deux oiseaux) ; supprimer deficiellement les lignes ferait perdre ces champs et rendrait le total d'effectifs dependant du sous-ensemble analyse",
    lignes=len(sans_mesures),
)
print("lignes sans mesures conservees :", len(sans_mesures))


[journal] mesures-absentes : conserver les lignes dans la table propre ; les analyses morphometriques utilisent dropna(subset=[...]) a l'usage
lignes sans mesures conservees : 2


In [8]:
# Sex inconnu pour 11 individus : incertitude d'observation, pas une erreur
sex_inconnu = brut[brut["Sex"].isna()]
print("sex inconnu :", len(sex_inconnu), "individus, repartis par espece :")
print(sex_inconnu["Species"].str.split().str[0].value_counts().to_string())

consigner(
    etape="sex-inconnu",
    probleme="Sex non renseigne pour 11 individus (terrain : determination impossible)",
    decision="conserver les lignes ; les analyses par sexe les excluent, le reste les garde",
    justification="l'absence de sexe n'invalide aucune mesure morphometrique ; une categorie 'inconnu' fausserait les moyennes par sexe, un filtre a l'usage est plus honnete",
    lignes=len(sex_inconnu),
)


sex inconnu : 11 individus, repartis par espece :


Species
Adelie    6
Gentoo    5
[journal] sex-inconnu : conserver les lignes ; les analyses par sexe les excluent, le reste les garde


**Un principe, trois réponses.** La section 3 a croisé trois statuts de manque et y a répondu trois fois, et c'est la colonne vertébrale du notebook : une valeur manquante ne se traite **jamais** par un `dropna()` global, parce que le **statut** de l'absence commande la réponse, pas l'inverse. `Comments` est **structurel** — la colonne n'est pas une variable analysable, elle documente des anomalies — donc on **retire la colonne**, sans perdre aucune observation. Les deux individus sans aucune mesure sont des **oiseaux réels** jamais mesurés — on **garde leurs lignes**, car leur espèce et leur île restent vraies, et ce sont précisément les analyses morphométriques qui les filtreront au moment du calcul, avec un `dropna(subset=[...])` ciblé. Les 11 sexes inconnos sont une **incertitude d'observation** — on **garde les lignes** et on laisse les analyses par sexe s'en charger.

Pour mesurer ce que cette prudence vaut, un chiffre brutal : un `dropna()` appliqué **globalement** à la table brute — « je retire toute ligne qui a **une** case vide » — efface **310 des 344** oiseaux, il n'en reste que 34. L'écrasante majorité de ce massacre tient à une seule colonne, `Comments`, quasi vide : une absence **structurelle** que nous avons très justement transformée en retrait de **colonne**, non en retrait de **lignes**. C'est tout l'enjeu du journal, condensé en une comparaison : **344 → 34** en un `dropna()` naïf, contre **344 → 344** dans notre table propre. Le **statut** de l'absence change tout — et c'est précisément ce que « **statuer** » signifie : décider, absence par absence, avant que la donnée ne soit perdue.

## 4. Doublons : exacts, de clé, et quasi

Trois vérifications d'escalade. Le **doublon exact** (toutes colonnes) est la corruption d'import classique. Le **doublon de clé** (la combinaison censée identifier une observation) révèle soit une vraie répétition, soit une clé mal choisie. Le **quasi-doublon** (même mesures, identités différentes) est le cas le plus subtil : deux oiseaux distincts peuvent peser pareil.


In [9]:
print("doublons exacts (toutes colonnes)      :", brut.duplicated().sum())
print("doublons (studyName, Sample Number)      :",
      brut.duplicated(subset=["studyName", "Sample Number"]).sum())
print("doublons (mesures becs+aile+masse)       :",
      brut.duplicated(subset=["Culmen Length (mm)", "Culmen Depth (mm)",
                              "Flipper Length (mm)", "Body Mass (g)"]).sum())


doublons exacts (toutes colonnes)      : 0
doublons (studyName, Sample Number)      : 124
doublons (mesures becs+aile+masse)       : 1


**Trois compteurs, trois questions.** Les trois chiffres de la cellule au-dessus — **0** doublon exact, **124** collisions de clé, **1** quasi-doublon — ne mesurent pas la même chose, et chacun envoie vers une enquête **différente**. C'est le point de méthode, identique à celui des manquants : un compte ne se **lit** pas, il se **suit**. Chaque chiffre pointe vers les lignes qu'il faut ouvrir. Le premier demande « y a-t-il eu une corruption d'import, une ligne enregistrée deux fois ? » ; le second, « ma clé identifiante est-elle trop grossière, ou y a-t-il de vraies répétitions ? » ; le troisième, « ces mesures identiques décrivent-elles un seul oiseau mesuré deux fois, ou deux oiseaux qui se ressemblent ? ». La cellule qui suit répond au second — et le quasi-doublon sera instruit juste après. Rien, ici, ne se tranche sur le seul nombre : ce sont toujours les **lignes** qui ont le dernier mot.

**Lecture.** Zéro doublon exact — le résultat est une **bonne nouvelle à écrire noir sur blanc**, pas un check inutile. Les 124 collisions `(studyName, Sample Number)` sont un **faux signal** : l'examen des lignes montre que `Sample Number` redémarre à 1 **pour chaque île au sein d'une même étude** (PAL0708 : Torgersen 1–20, Dream 1–50, Biscoe 1–34) — les collisions vivent donc toutes au sein d'une même étude, le plus souvent entre îles différentes, et la clé identifiante réelle est `(studyName, Sample Number, Individual ID)`. Leçon : un doublon de clé se confirme en regardant **les lignes** collisions, jamais sur le seul compte. Reste le quasi-doublon unique — à instruire ci-dessous.

In [10]:
# Instruire le quasi-doublon : deux lignes, quatre mesures "identiques" — memes oiseaux ?
mesures = ["Culmen Length (mm)", "Culmen Depth (mm)", "Flipper Length (mm)", "Body Mass (g)"]
quasi = brut[brut.duplicated(subset=mesures, keep=False)]
quasi[["studyName", "Sample Number", "Individual ID", "Island", "Sex"] + mesures]


,studyName,Sample Number,Individual ID,Island,Sex,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g)
3,PAL0708,4,N2A2,Torgersen,NaN,NaN,NaN,NaN,NaN
271,PAL0910,120,N38A2,Biscoe,NaN,NaN,NaN,NaN,NaN


In [11]:
verdict = "deux individus distincts" if quasi["Individual ID"].nunique() == len(quasi) else "doublon probable"
consigner(
    etape="quasi-doublon",
    probleme="1 paire de lignes aux 4 mesures toutes manquantes (NaN — faux quasi-doublon fabrique par duplicated)",
    decision=f"conserver les deux lignes ({verdict})",
    justification="mesures toutes NaN : duplicated() les traite comme egales ; Individual ID differents — deux oiseaux reels jamais mesures. Supprimer serait inventer une erreur de saisie",
    lignes=len(quasi),
)
print("verdict :", verdict)


[journal] quasi-doublon : conserver les deux lignes (deux individus distincts)
verdict : deux individus distincts


**Quatre mesures égales, deux oiseaux.** Le quasi-doublon unique est un **piège** — et c'est le plus instructif de tous. Les deux lignes qu'il met en face ont leurs **quatre** mesures au NaN, et `duplicated()` traite le NaN comme égal au NaN : d'où le « doublon » sur les mesures. Mais leurs `Individual ID` diffèrent (`N2A2` vs `N38A2`) et leurs îles aussi (Torgersen vs Biscoe) : ce ne sont pas un oiseau mesuré deux fois, ce sont **les mêmes deux oiseaux jamais mesurés** qu'on a déjà rencontrés en section 3. Le « quasi-doublon » n'est donc pas une erreur de saisie à corriger, c'est un **artefact** de la façon dont la détection de doublons se comporte sur des données vides. Le verdict de la cellule au-dessus — « deux individus distincts, on garde les deux » — est exactement le bon : supprimer l'une des deux lignes, ce serait **inventer** une faute de saisie qui n'existe pas. Leçon : un doublon signalé sur des valeurs manquantes se statue **toujours** sur les identifiants, jamais sur les mesures.

## 5. Types et catégories : rendre la table lisible

Deux transformations de fond, sans perte d'information. Les **noms d'espèces verbeux** (`"Adelie Penguin (Pygoscelis adeliae)"`) se réduisent au nom d'usage (premier mot) — le nom scientifique reste documenté dans `data/PROVENANCE.md`. La **date de ponte en texte** devient un vrai `datetime`, d'où l'on extraira l'année. Et le dictionnaire de renommage traduit les colonnes vers les noms compacts utilisés par la version de référence du jeu (`species`, `bill_length_mm`, …).

Aucune de ces trois transformations ne **perd** d'information : on ne supprime ni ligne ni valeur, on **traduit**. C'est la différence entre *nettoyer* (rendre la table lisible et comparable) et *modifier les données* (changer ce qu'elles disent). On vérifiera cette absence de perte à la section 7, quand la table reconstruite rejoindra la référence **ligne pour ligne**.

In [12]:
# Nom court d'espece + date reelle
brut = brut.assign(
    espece=brut["Species"].str.split().str[0],
    date_ponte=pd.to_datetime(brut["Date Egg"]),
)
print(brut["espece"].value_counts().to_string())
print("annees de terrain :", sorted(brut["date_ponte"].dt.year.unique()))

consigner(
    etape="categories",
    probleme="Species verbeux ('Adelie Penguin (Pygoscelis adeliae)'), Date Egg stockee en texte, Sex en MAJUSCULES",
    decision="colonne espece = premier mot du nom scientifique ; date_ponte en datetime ; Sex en minuscules",
    justification="noms d'usage comparables a la reference ; une date texte ne se trie ni ne se resample pas ; casse uniforme pour eviter 'MALE' != 'Male'",
    lignes=344,
)
brut["Sex"] = brut["Sex"].str.lower()


espece
Adelie       152
Gentoo       124
Chinstrap     68
annees de terrain : [np.int32(2007), np.int32(2008), np.int32(2009)]
[journal] categories : colonne espece = premier mot du nom scientifique ; date_ponte en datetime ; Sex en minuscules


**Lecture — le texte devient une date.** La colonne `Date Egg` arrivait en **texte** — « 2007-11-16 », une chaîne qu'aucun calcul ne touche. `pd.to_datetime` en fait un vrai `datetime`, et c'est tout l'usage qui s'ouvre : la cellule affiche les **années de terrain** `[2007, 2008, 2009]`, qu'on vient d'extraire en une ligne avec `.dt.year` — impossible sur une simple chaîne. Un texte ne se **trie** pas chronologiquement, ne se **resample** pas par saison, ne se **compare** pas à une autre date. La conversion n'est pas un luxe cosmétique : c'est ce qui rendra possible, dès le notebook 1.6, toute analyse qui croise le temps et la biologie. C'est aussi un cas type de transformation **sans perte** : on n'a changé ni la valeur ni le nombre de lignes, seulement la **nature** du champ — et la section 7 le prouvera.

In [13]:
# Renommage vers les noms compacts de la version de reference
brut = brut.rename(columns={
    "Island": "island",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
    "espece": "species",
})
propre = brut[["species", "island", "bill_length_mm", "bill_depth_mm",
               "flipper_length_mm", "body_mass_g", "sex", "date_ponte"]].copy()
propre["year"] = propre["date_ponte"].dt.year
propre = propre.drop(columns=["date_ponte"])
propre.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


**Reformater pour la machine.** Les transformations de la section 5 ne ressemblent à du nettoyage que de loin : on ne jette rien, on **traduit**. Le nom d'espèce verbeux (`Adelie Penguin (Pygoscelis adeliae)`) devient son premier mot — `Adelie`, `Gentoo`, `Chinstrap` — des libellés courts qu'on peut croiser, grouper et comparer sans se battre avec 38 caractères. La date de ponte, stockée en **texte**, devient un vrai `datetime` : c'est ce qui rendra possibles, au notebook suivant, le tri chronologique, l'extraction de l'année et de la saison, le resamplement — rien de tout cela ne se fait sur une chaîne. Le sexe passe en minuscules pour que `MALE` ne puisse plus glisser à côté de `Male` comme deux catégories distinctes. Et le renommage des colonnes vers les noms compacts de la référence (`bill_length_mm`, `body_mass_g`…) n'est pas cosmétique : c'est ce qui permettra, en section 7, d'aligner **notre** table sur **la** table des auteurs, colonne pour colonne. Reformater pour la machine, c'est poser les conditions du notebook suivant — celui-ci prépare le terrain que les statistiques vont empieter.

## 6. Valeurs extrêmes : examiner, pas purger

La règle de l'écart interquartile (IQR) signale les points au-delà de 1,5 fois l'écart interquartile au-dessus du troisième quartile ou en dessous du premier. Mais **signaler n'est pas décider** : sur des mesures biologiques, un manchot lourd est un manchot lourd, pas une erreur de saisie. On examine, puis on décide — et la décision entre au journal.

Pourquoi ne pas, plus simplement, `dropna()` tout ce qui sort des bornes ? Parce que sur des mesures biologiques, l'extrême **est** le signal : un oiseau plus lourd que la fourchette est un oiseau réel, et le supprimer **biaiserait silencieusement** toutes les moyennes qui suivront. La règle IQR est un **cri d'alerte**, pas une sentence. On l'écoute, on ouvre les cas qu'elle pointe, et c'est le **jugement de domaine** — pas le seuil — qui tranche.

In [14]:
q1, q3 = propre["body_mass_g"].quantile([0.25, 0.75])
iqr = q3 - q1
borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
extremes = propre[(propre["body_mass_g"] < borne_basse) | (propre["body_mass_g"] > borne_haute)]
print(f"IQR masse : [{q1:.0f}, {q3:.0f}] -> bornes [{borne_basse:.0f}, {borne_haute:.0f}]")
print("valeurs extremes selon la regle :", len(extremes), "| plage observee :",
      propre["body_mass_g"].min(), "-", propre["body_mass_g"].max(), "g")


IQR masse : [3550, 4750] -> bornes [1750, 6550]
valeurs extremes selon la regle : 0 | plage observee : 2700.0 - 6300.0 g


In [15]:
consigner(
    etape="extremes",
    probleme="regle IQR appliquee a la masse corporelle",
    decision="aucune ligne retiree : aucune valeur hors bornes, plage 2700-6300 g biologiquement plausible",
    justification="l'examen precede la purge ; ici meme la regle la plus courante ne signale rien a retirer — le constat se consigne, il vaut preuve",
    lignes=0,
)


[journal] extremes : aucune ligne retiree : aucune valeur hors bornes, plage 2700-6300 g biologiquement plausible


**L'IQR propose, le domaine dispose.** La règle de l'écart interquartile est le test automatique d'extrêmes le plus répandu, et ici elle **ne propose rien** : la masse observée (2700 à 6300 g) reste à l'intérieur des bornes [1750, 6550] qu'elle calcule, zéro point au-delà. Le point n'est pas ce zéro — c'est l'**ordre** des opérations qu'il autorise. Sur des mesures biologiques, un manchot plus lourd que la fourchette est un **vrai** manchot lourd, pas une coquille : une purge automatique l'aurait supprimé et biaisé, sans bruit, toutes les moyennes qui suivent. D'où la règle du carnet — **examiner, pas purger** : le seuil **propose** une liste de suspects, le **jugement de domaine** **dispose** de chacun, et la décision entre au journal. Conséquence contre-intuitive mais saine : le résultat « **rien à retirer** » n'est pas un non-événement, c'est un **résultat à consigner** exactement comme le serait un retrait — car c'est la preuve qu'on a **regardé** avant d'agir. La section 7 confirmera que cette prudence ne coûte rien : la référence, elle aussi, garde toutes ces valeurs.

## 7. La référence existe : comparer

Ce jeu a une version de référence remise en forme par les auteurs (`data/penguins.csv`). La nôtre doit la rejoindre — c'est le test de fin de nettoyage : mêmes dimensions, mêmes effectifs par espèce, mêmes valeurs. La comparaison n'est pas une formalité : elle **prouve** que le nettoyage n'a ni perdu ni fabriqué de lignes.

C'est la forme la plus forte de validation d'un nettoyage, et de loin : on ne se contente pas de dire « ça a l'air cohérent », on montre que la table qu'on a **reconstruite de zéro** depuis l'export brut est **identique** à celle que les auteurs ont publiée et qu'on peut télécharger **indépendamment**. Deux pipelines, deux équipes, un résultat byte pour byte : c'est la preuve que nos décisions n'ont ni perdu ni inventé rien.

In [16]:
reference = pd.read_csv("../data/penguins.csv")

print("notre table   :", propre.shape, "| reference :", reference.shape)
print("effectifs par espece (notre / reference) :")
comparaison = pd.concat(
    [propre["species"].value_counts().rename("notre nettoyage"),
     reference["species"].value_counts().rename("reference")], axis=1)
comparaison["egaux"] = comparaison["notre nettoyage"] == comparaison["reference"]
comparaison


notre table   : (344, 8) | reference : (344, 8)
effectifs par espece (notre / reference) :


,notre nettoyage,reference,egaux
species,,,
Adelie,152,152,True
Gentoo,124,124,True
Chinstrap,68,68,True


In [17]:
# Comparaison ligne a ligne sur les 4 mesures (l'ordre des lignes est conserve)
for colonne in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
    a, b = propre[colonne], reference[colonne]
    divergentes = ((a != b) & ~(a.isna() & b.isna())).sum()
    print(f"{colonne:20s} : {divergentes} ligne(s) divergente(s)")

print("ordre des lignes conserve :", (propre.index == reference.index).all())


bill_length_mm       : 0 ligne(s) divergente(s)
bill_depth_mm        : 0 ligne(s) divergente(s)
flipper_length_mm    : 0 ligne(s) divergente(s)
body_mass_g          : 0 ligne(s) divergente(s)
ordre des lignes conserve : True


**La référence comme témoin.** On a **reconstruit de zéro** la table que les auteurs du jeu ont publiée, en partant de l'export brut, et elle **coïncide** : mêmes dimensions (344 × 8), mêmes effectifs par espèce (152 / 124 / 68), **zéro** ligne divergente sur les quatre mesures, et l'ordre des lignes conservé. C'est la forme la plus forte de validation d'un nettoyage qu'on puisse obtenir : on ne se contente pas de dire « ça a l'air cohérent », on montre que **deux pipelines indépendants** — le nôtre, et celui des auteurs, téléchargeable séparément — produisent **le même résultat byte pour byte**. C'est la preuve formelle que nos décisions n'ont ni **perdu** ni **fabriqué** une seule ligne. Et la boucle se referme sur la section 3 : la référence **garde** les mêmes valeurs manquantes que nous, ce qui confirme a posteriori que « la table propre **garde**, l'analyse **filtre** » était le bon réflexe — on aurait dévié de la référence du jour même où l'on aurait supprimé un manquant au lieu de le laisser vivre.

**Lecture.** Le nettoyage rejoint la référence : mêmes dimensions, mêmes effectifs, et aucune ligne divergente sur les quatre mesures morphométriques (le code ci-dessus le mesure) — la version de référence conservait les mêmes valeurs manquantes que nous (c'est la confirmation du choix de la section 3 : **la table propre garde, l'analyse filtre**). Un détail de méthode : la comparaison se fait ligne à ligne sur l'ordre conservé, pas par un `merge` sur les colonnes nominales — `sex` contient des NaN qui ne se joignent pas à eux-mêmes et un croisement démultiplierait les lignes. Notre pipeline a reproduit, décision par décision, le travail des auteurs du jeu.

## Exercices

Quatre exercices sur la table `propre`. Les cellules s'exécutent telles quelles ; remplacez les `None`/`pass` en suivant les étapes. Chacun met en œuvre **une** des décisions que le carnet a prises à votre place, pour que vous la preniez **vous-même** : l'audit des isotopes (section 3), l'instruction d'un doublon de clé (section 4), l'usage de la date en années (section 5), et la sensibilité d'un filtre (l'idée maîtresse « nettoyer n'est pas supprimer »). La consigne commune : ne pas **deviner** le résultat, le **produire** — puis, pour les exercices 2 et 4, **l'interpréter** en une phrase. C'est ce passage du geste à l'interprétation qui fait la différence entre avoir simplement suivi le code et l'avoir compris.

In [18]:
journal_df = pd.DataFrame(journal)
journal_df[["etape", "probleme", "decision", "lignes_concernees"]]


,etape,probleme,decision,lignes_concernees
0,comments,Comments quasi vide (la note n'existe que pour...,retirer la colonne de la table de travail,344
1,mesures-absentes,"2 individus sans aucune mesure (bec, aile, masse)",conserver les lignes dans la table propre ; le...,2
2,sex-inconnu,Sex non renseigne pour 11 individus (terrain :...,conserver les lignes ; les analyses par sexe l...,11
3,quasi-doublon,1 paire de lignes aux 4 mesures toutes manquan...,conserver les deux lignes (deux individus dist...,2
4,categories,Species verbeux ('Adelie Penguin (Pygoscelis a...,colonne espece = premier mot du nom scientifiq...,344
5,extremes,regle IQR appliquee a la masse corporelle,aucune ligne retiree : aucune valeur hors born...,0


**Le journal est le livrable.** Cinq entrées, une par transformation, dans l'ordre où les décisions ont été prises — et c'est **ce tableau**, pas le code, qui est le livrable auditable du notebook. Chaque ligne nomme le **problème** observé, la **décision** prise, la **justification**, et le **compte de lignes** concernées ; et chaque compte est **reproductible** en re-exécutant la cellule juste au-dessus. C'est ici que vit la promesse de reproductibilité du carnet : un lecteur qui audite le journal peut, pour chaque ligne, remonter à la mesure qui a motivé la décision, rejouer la transformation, et constater que la table propre ne diffère de la table brute **que** par ces cinq gestes documentés — ni plus, ni moins. Le code prouve **comment** on a fait ; le journal prouve **pourquoi** on l'a fait et **à quel coût**, en lignes. Un nettoyage dont on ne peut pas produire le journal est un nettoyage qu'on ne peut pas reproduire — et c'est tout l'enjeu.

## Exercices

Quatre exercices sur la table `propre`. Les cellules s'exécutent telles quelles ; remplacez les `None`/`pass` en suivant les étapes.


### Exercice 1 : l'audit des isotopes

Les colonnes `Delta 15 N (o/oo)` et `Delta 13 C (o/oo)` ont été retirées de la table de travail sans décision au journal. Comptez leurs manquants sur `brut` (avant renommage, rechargez `data/penguins_raw.csv` si besoin), puis consignez la décision au journal avec votre justification.

C'est l'occasion de faire vous-même le geste de la section 3 : les isotopes sont un **quatrième** type de manque (échantillonnage partiel — un sous-ensemble d'oiseaux n'a pas été prélevé) que le carnet a consigné à votre place. Le comptage tient en une ligne ; la **justification** est l'exercice. Un bon « pourquoi » nomme le **statut** de l'absence (programmé par le protocole de terrain, pas une donnée perdue) et dit ce que le retrait **ne change pas** (aucune mesure morphométrique n'en dépend).

In [19]:
# Exercice 1 : audit isotopes
brut_exo = pd.read_csv("../data/penguins_raw.csv")
# Etape 1: compter les NA des deux colonnes Delta
delta_na = None  # TODO: brut_exo[["Delta 15 N (o/oo)", "Delta 13 C (o/oo)"]].isna().sum()
# Etape 2: consigner la decision (retirer de la table de travail, garder la trace)
# consigner("isotopes", ..., ..., ..., lignes=...)   # TODO: votre justification
print("Exercice a completer")


Exercice a completer


### Exercice 2 : le doublon de clé, instruit

Les 124 collisions `(studyName, Sample Number)` ont été écartées comme faux signal. Vérifiez-le vous-même : affichez 3 paires de lignes en collision et constatez que les `Individual ID` diffèrent. Rédigez ensuite la phrase de verdict dans le `print` final (une ligne suffit).


In [20]:
# Exercice 2 : instruire le doublon de cle
brut_exo2 = pd.read_csv("../data/penguins_raw.csv")
# Etape 1: lignes dupliquees sur (studyName, Sample Number), keep=False pour voir les paires
collisions = None  # TODO: brut_exo2[brut_exo2.duplicated(subset=["studyName", "Sample Number"], keep=False)]
# Etape 2: afficher 3 exemples de paires (head(6)) et comparer les Individual ID
pass  # TODO
print("Exercice a completer : verdict a rediger")


Exercice a completer : verdict a rediger


### Exercice 3 : l'année de ponte

Depuis `propre`, comptez les pontes par année (`value_counts`), puis la masse moyenne par année. Les effectifs sont-ils équilibrés entre les trois années de terrain ?

Cet exercice rend concret ce que la section 5 a préparé : sans la conversion **texte → `datetime`**, l'année ne serait qu'un sous-texte qu'on extrairait à la main. Compter les pontes par année, c'est **utiliser** la date pour croiser le temps et la biologie. La question finale — les effectifs sont-ils équilibrés entre 2007, 2008 et 2009 ? — est celle que pose tout analyste **avant** de comparer des moyennes entre années : comparer des moyennes calculées sur des effectifs très déséquilibrés, c'est risquer de comparer du bruit.

In [21]:
# Exercice 3 : analyses par annee
# Etape 1: pontes par annee
par_annee = None  # TODO: propre["year"].value_counts().sort_index()
# Etape 2: masse moyenne par annee (dropna implicite des lignes sans mesure)
masse_annee = None  # TODO: propre.groupby("year")["body_mass_g"].mean()
print("Exercice a completer")


Exercice a completer


### Exercice 4 : la sensibilité du filtre

Calculez la masse moyenne des Gentoo **avec** puis **sans** `dropna` sur la masse. Que constatez-vous — et pourquoi le filtre ne change-t-il rien au résultat ?

C'est l'exercice qui verrouille l'idée maîtresse du carnet : **nettoyer n'est pas supprimer**. Si les deux moyennes coïncident, c'est que `mean()` **ignore déjà** les NaN — le `dropna` explicite est alors redondant, et la table propre a tout son intérêt à **conserver** ces lignes pour les analyses qui, elles, n'ont pas besoin de la masse (compter les Gentoo, par exemple). Le résultat attendu — les deux valeurs égales — **est** la leçon : le filtre se pose **à l'usage**, pas au moment du nettoyage.

In [22]:
# Exercice 4 : sensibilite du filtre dropna
gentoo = propre[propre["species"] == "Gentoo"]
# Etape 1: moyenne brute (les NaN sont ignores par mean())
avec = None  # TODO: gentoo["body_mass_g"].mean()
# Etape 2: moyenne apres dropna explicite
sans = None  # TODO: gentoo["body_mass_g"].dropna().mean()
print("Exercice a completer : comparez et expliquez")


Exercice a completer : comparez et expliquez


## Conclusion

Ce notebook a mené l'audit complet d'un jeu réel : dimensions, types, manquants **à trois statuts** (structurel, incertitude, donnée perdue), doublons **à trois niveaux** (exact, de clé, quasi), catégories à normaliser, dates à convertir, extrêmes à examiner avant toute purge. Le fil conducteur : **chaque décision consignée**, jusqu'au constat « rien à retirer » qui se prouve comme les autres. La table propre rejoint la référence des auteurs — et elle a gardé ses valeurs manquantes, parce que nettoyer n'est pas supprimer.

**Ce qu'on emporte.** La méthode est transférable à n'importe quel jeu, et elle tient en quatre verbes : **auditer** (les cinq sondes — dimensions, types, manquants, doublons, plages — forment une check-list complète, et chaque décision à venir s'y rattache), **statuer** (le **statut** de chaque anomalie commande la réponse, jamais un `dropna` global), **consigner** (le journal rend le nettoyage auditable et reproductible), et **comparer** (rejoindre une référence **indépendante** est la preuve la plus forte qu'on n'a ni perdu ni inventé). C'est cet enchaînement — pas les lignes de code — qui fait de vous un agent capable d'orchestrer un nettoyage honnête. Et parce que chaque étape est **consignée** au journal, le lecteur — ou l'agent que vous orchestrerez — peut **rejouer** le nettoyage de bout en bout, décision par décision, et y arriver au même résultat : c'est la **reproductibilité au sens fort** qu'un journal bien tenu procure, au-delà du simple « le code tourne ». C'est ce qui fait passer le carnet du statut de **démonstration** à celui d'**outil** : un nettoyage qu'on ne peut pas rejouer n'est qu'une anecdote, un nettoyage qu'on rejoue et qui reproduit la référence est une méthode.

**Suite logique** : [1.6 Statistiques descriptives](1.6-Statistiques_Descriptives.ipynb) — résumer ces données par des chiffres : tendance centrale, dispersion, distributions, corrélation.

## Références

1. H. Wickham, « Tidy Data », *Journal of Statistical Software* 59(10), 2014 — la forme propre d'une table.
2. pandas docs, « Working with missing data » et « Duplicate data », pandas.pydata.org.
3. Station Palmer LTER / K. Gorman — données Penguin size clutches (2007-2009), voir `data/PROVENANCE.md`.
